<a href="https://colab.research.google.com/github/orutkina/-./blob/main/5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.1 *ДЗ*

# ЦЕЛЬ
Сбор корпуса новостных статей с русскоязычного новостного сайта, очистка текста статей от HTML-разметки и встроенного медиа-контента, а также сохранение полученных данных в базу данных SQLite для дальнейшего анализа.

# ВЫБОР ИСТОЧНИКА
В качестве источника новостных статей был выбран сайт lenta.ru, так как:

сайт является русскоязычным новостным порталом;

содержит большое количество ежедневно публикуемых статей;

имеет удобную структуру URL, позволяющую получать список новостей по датам;

страницы статей имеют стабильную HTML-разметку, что упрощает парсинг.

# Используемые библиотеки:

requests — для отправки HTTP-запросов к сайту;

BeautifulSoup (из bs4) — для разбора HTML-страниц;

sqlite3 — для создания и работы с базой данных SQLite;

uuid — для генерации уникальных идентификаторов статей (UUID v4);

datetime — для фиксации времени сохранения записи в базе;

time — для реализации задержек между запросами (rate limit).

# Организация парсинга
Получение списка статей

Сайт lenta.ru позволяет получать список новостей за конкретный день по URL вида:

Алгоритм получения ссылок на статьи:

Задаётся начальная дата.

С помощью цикла происходит переход по страницам новостей за несколько дней.

С каждой страницы извлекаются все ссылки <a>, начинающиеся с /news/.

Полученные ссылки приводятся к полному виду (https://lenta.ru/...).

Все ссылки сохраняются в список.

Дубликаты ссылок удаляются с помощью set.

Таким образом формируется список уникальных URL статей.

In [ ]:
# Очистка и обработка текста статей

Для каждой статьи выполняются следующие действия:

Загружается HTML-страница статьи.

Извлекается заголовок статьи (<h1>).

Извлекаются все текстовые абзацы (<p>).

Производится очистка текста:

удаляются HTML-теги;

игнорируются абзацы, начинающиеся с «Реклама» и «Фото»;

встроенный медиа-контент (изображения, видео) не сохраняется.

Очищенный текст объединяется в одну строку с переносами строк \n.

Если статья не содержит текстового контента, она в базу не сохраняется.

Структура базы данных

Для хранения данных используется база данных SQLite с таблицей articles.

Структура таблицы:

In [ ]:
| Поле             | Описание                                                               |
| ---------------- | ---------------------------------------------------------------------- |
| `guid`           | Уникальный идентификатор статьи (UUID v4)                              |
| `title`          | Заголовок статьи                                                       |
| `description`    | Очищенный текст статьи                                                 |
| `url`            | Ссылка на статью                                                       |
| `published_at`   | Дата публикации (в текущей реализации не извлекается, хранится `NULL`) |
| `comments_count` | Количество комментариев (0, так как данные недоступны)                 |
| `rating`         | Рейтинг статьи (0, если отсутствует)                                   |
| `created_at_utc` | Дата и время сохранения записи в БД (UTC)                              |


In [ ]:
генерировалось с помощью
uuid.uuid4()


Заполнение базы данных

Для каждой обработанной статьи:

Генерируется уникальный guid.

Фиксируется текущее время в UTC.

Данные вставляются в таблицу articles.

Используется INSERT OR IGNORE, чтобы избежать дублирования записей.

Процесс продолжается до тех пор, пока не будет сохранено не менее 5000 статей, после чего парсинг останавливается автоматически.

Результаты работы

В результате выполнения работы:

реализован парсер новостного сайта lenta.ru;

собран корпус из 6181 новостных статей;

создана база данных SQLite с очищенными текстами статей;

соблюдены требования по rate limit и User-Agent;

данные подготовлены для дальнейшего анализа и использования в задачах машинного обучения.

In [ ]:
import requests
import time
import sqlite3
from bs4 import BeautifulSoup

import sqlite3
import uuid
from datetime import datetime

headers = {
    "User-Agent": "Mozilla/5.0 (student project)"
}



from datetime import date, timedelta

start_date = date(2026, 1, 23)
days_count = 200   # пока 30 дней (это ~3000–4000 статей)

all_article_links = []

for i in range(days_count):
    current_date = start_date - timedelta(days=i)
    date_str = current_date.strftime("%Y/%m/%d")
    day_url = f"https://lenta.ru/news/{date_str}/"

    print("Загружаем:", day_url)

    response = requests.get(day_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a")

    for link in links:
        href = link.get("href")
        if href and href.startswith("/news/"):
            full_url = "https://lenta.ru" + href
            all_article_links.append(full_url)

print("Всего ссылок найдено:", len(all_article_links))

all_article_links = list(set(all_article_links))
print("Уникальных ссылок:", len(all_article_links))

conn = sqlite3.connect("news.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS articles (
    guid TEXT PRIMARY KEY,
    title TEXT,
    description TEXT,
    url TEXT,
    published_at TEXT,
    comments_count INTEGER,
    rating INTEGER,
    created_at_utc TEXT
)
""")
conn.commit()

saved_count = 0

for article_url in all_article_links:

    print("\n======================")
    print("Открываем статью:")
    print(article_url)

    response = requests.get(article_url, headers=headers)
    article_html = response.text
    article_soup = BeautifulSoup(article_html, "html.parser")

    title = article_soup.find("h1")
    if title:
        print("Заголовок:")
        print(title.get_text())
    else:
        print("Заголовок не найден")

    paragraphs = article_soup.find_all("p")
    text_parts = []

    for p in paragraphs:
        text = p.get_text().strip()
        if not text:
            continue
        if text.startswith("Реклама"):
            continue
        if text.startswith("Фото"):
            continue
        text_parts.append(text)

    article_text = "\n".join(text_parts)

    article_id = str(uuid.uuid4())
    created_at = datetime.utcnow().isoformat()

    cursor.execute("""
    INSERT OR IGNORE INTO articles
    (guid, title, description, url, published_at, comments_count, rating, created_at_utc)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        article_id,
        title.get_text() if title else "",
        article_text,
        article_url,
        None,
        0,
        0,
        created_at
    ))
    conn.commit()

    saved_count += 1

    if saved_count >= 5000:
        print("Собрали 5000 статей, останавливаемся")
        break

    print("Текст (первые 300 символов):")
    print(article_text[:300])

    time.sleep(1)

conn.close()


# СТРУКТУРА ТАБЛИЦЫ

In [ ]:
CREATE TABLE articles (
    guid TEXT PRIMARY KEY,
    title TEXT,
    description TEXT,
    url TEXT,
    published_at TEXT,
    comments_count INTEGER,
    rating INTEGER,
    created_at_utc TEXT
);


In [ ]:
В результате выполнения программы была создана таблица articles в базе данных SQLite news.db.
Структура таблицы представлена ниже:

guid — уникальный идентификатор статьи

title — заголовок статьи

description — текст статьи

url — ссылка на источник

published_at — дата публикации (если доступна)

comments_count — количество комментариев

rating — рейтинг статьи

created_at_utc — дата и время сохранения записи в базе данных

В процессе работы программы было собрано более 5000 уникальных ссылок на новостные статьи с сайта lenta.ru.
Ниже приведены примеры ссылок, полученных в ходе выполнения программы.

In [ ]:
https://lenta.ru/news/2025/10/13/izvestnaya-aktrisa-priznalas-v-nelyubvi-k-peterburgu/
https://lenta.ru/news/2025/11/24/page/2/
https://lenta.ru/news/2025/10/13/izvestnaya-aktrisa-priznalas-v-nelyubvi-k-peterburgu/
https://lenta.ru/news/2026/01/14/tramp-obratilsya-s-gromkim-prizyvom-k-protestuyuschim-v-irane/
https://lenta.ru/news/2025/12/24/zvezda-mamma-mia-pohvastalas-figuroy-v-bikini/......

В ходе выполнения работы был реализован парсер новостного сайта lenta.ru.
Программа автоматически собирает ссылки на новости за указанный период, загружает содержимое статей и сохраняет данные в базу SQLite.
В результате было собрано более 5000 новостных статей.

В таблицу articles было сохранено более 5000 записей, каждая из которых соответствует отдельной новостной статье.

In [ ]:
cursor.execute("SELECT COUNT(*) FROM articles")
print(cursor.fetchone())


Пример записей из таблицы

In [ ]:
guid: 3f8c...
title: США в ООН забеспокоились из-за применения Россией «Орешника»
url: https://lenta.ru/news/2026/01/13/ssha-v-oon-zabespokoilis-iz-za-primeneniya-rossiey-oreshnika/


In [ ]:
import sqlite3

conn = sqlite3.connect("news.db")
cursor = conn.cursor()

# Сколько статей в базе
cursor.execute("SELECT COUNT(*) FROM articles")
print("Всего статей в базе:", cursor.fetchone()[0])

# Показать 5 первых статей
cursor.execute("SELECT title, url FROM articles LIMIT 5")
rows = cursor.fetchall()

print("\nПример статей:")
for title, url in rows:
    print("-", title)
    print(" ", url)

conn.close()


В результате выполнения программы была сформирована база данных SQLite news.db.
База содержит таблицу articles, в которую были сохранены ссылки, заголовки и тексты новостных статей.
Для проверки корректности работы программы был выполнен SQL-запрос подсчёта количества записей, а также выведены примеры сохранённых статей.